In [ ]:
import mlflow
import ollama
from loguru import logger
from openai import OpenAI
from mlflow.entities import SpanType
import time

In [3]:
config_1 = {
    "version": "1.0",
    "model": "gpt-oss:20b",
    "temperature": 0.1,
    "max_tokens": 100
}

config_2 = {
    "version": "2.0",
    "model": "gpt-oss:20b",
    "temperature": 0.5,
    "max_tokens": 100
}

In [4]:
PORT = "5001"
EXPERIMENT_NAME = "compare_models_experiment"
TRACE_ID = "id-12345"
TRACE_KEY = "model_comparison"
TRACE_VALUE = "1.0"

OLLAMA_API_URL = "http://localhost:11434/v1"
LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

In [5]:
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1774103933537, experiment_id='2', last_update_time=1774103933537, lifecycle_stage='active', name='compare_models_experiment', tags={}, workspace='default'>

In [ ]:
from mlflow.genai.scorers import RelevanceToQuery, Safety
import pandas as pd
import os

os.environ["OPENAI_API_KEY"] = "dummy"
os.environ["OPENAI_API_BASE"] = "http://localhost:11434/v1"
llm_as_judge = "openai:/gpt-oss:20b"

class LLMClient(mlflow.pyfunc.PythonModel):

    def __init__(self, config):
        self.config = config
        self.client = self._set_llm(OLLAMA_API_URL)

    def _set_llm(self, url):
        try:
            logger.info(f"Ollama API URL: {url}")
            return OpenAI(base_url=url, 
                          api_key="dummy")
        except Exception as e:
            print(f"Error initializing OpenAI client: {e}")
            return None

    @mlflow.trace(span_type=SpanType.CHAIN)
    def __call__(self, user_input):
        if self.client is None:
            return "LLM client not initialized."
        try:
            response = self.client.chat.completions.create(
            model=self.config["model"],
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": user_input}
            ],
            temperature=self.config["temperature"],
            max_tokens=self.config["max_tokens"]
            )
            return response.choices[0].message.content

        except Exception as e:
            print(f"Error during LLM call: {e}")
            return "Sorry, I encountered an error."
        
eval_data = pd.DataFrame([
    {"inputs": {"query": "What is MLflow?"}},
    {"inputs": {"query": "How do I track experiments?"}},
])


for config in [config_1, config_2]:

    llm_client = LLMClient(config)

    def predict(query: str) -> str:
            return llm_client(query)

    with mlflow.start_run(run_name=f"LLM Evaluation - {config['version']}"):

        mlflow.log_params(config)

        results = mlflow.genai.evaluate(
                                        data=eval_data,
                                        predict_fn=predict,
                                        scorers=[RelevanceToQuery(model=llm_as_judge),
                                                 Safety(model=llm_as_judge)]
                                                )

2026-03-21 17:01:42.734 | INFO     | __main__:_set_llm:17 - Ollama API URL: http://localhost:11434/v1
2026/03/21 17:01:42 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/03/21 17:01:42 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:54, Remaining: 00:00] 


2026-03-21 17:02:42.117 | INFO     | __main__:_set_llm:17 - Ollama API URL: http://localhost:11434/v1
2026/03/21 17:02:42 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:55, Remaining: 00:00] 
